# 04 — Amostragem e Experimentos

**Objetivos de aprendizagem:** diferenciar amostragens, identificar viés e planejar experimentos

**Pré-requisitos:** Fundamentos e probabilidade básica

**Origem:** Slides, aulas 05–06, páginas 120–145. Conteúdos adicionais estão identificados como **Aprofundamento**.

**Convenção:** `datasets/` reúne dados de entrada; `outputs/` reúne resultados. Dados inventados para demonstrações são sempre identificados como *sintéticos*.

### Materiais complementares

- [Guia técnico: Amostragem e Experimentos](./reference_materials/04_amostragem_e_experimentos.pdf)
- [Cheat sheet: Amostragem e Experimentos](./cheat_sheets/cheat_sheet_amostragem_e_experimentos.pdf)


## Preparação do ambiente

Execute a célula abaixo. Os notebooks são independentes e não dependem de variáveis definidas em outros arquivos.

> **Resultados reproduzíveis:** gráficos PNG e tabelas CSV selecionados são gravados automaticamente em `outputs/` ao executar as células na ordem. Os arquivos em `datasets/` são preservados. Esta disciplina usa Matplotlib, não o renderer `iframe` do Plotly.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Execute o JupyterLab com o diretório desta disciplina como raiz.
BASE = Path.cwd()
DATASETS = BASE / "datasets"
OUTPUTS = BASE / "outputs"
DATASETS.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

def ler_dataset(nome):
    caminho = DATASETS / nome
    if not caminho.is_file():
        raise FileNotFoundError(f"Dataset não encontrado: {caminho}. Abra o JupyterLab na raiz da disciplina.")
    return pd.read_csv(caminho)

RNG = np.random.default_rng(42)

## 1. Amostragem probabilística

Na amostragem aleatória simples, cada subconjunto de tamanho fixo tem a mesma probabilidade. Sistemática, estratificada e por conglomerados exigem desenhos e ponderações próprios. Uma amostra grande não corrige viés de seleção.

In [2]:
pop=np.r_[RNG.normal(10,2,800),RNG.normal(18,2,200)] # população sintética
simple=RNG.choice(pop,100,replace=False)
seletiva=RNG.choice(pop[800:],100,replace=True) # seleção enviesada intencional
print('média população',pop.mean(),'aleatória',simple.mean(),'enviesada',seletiva.mean())

média população 11.542216898008109 aleatória 11.597781019231315 enviesada 17.367020595928288


## 2. Estratos e ponderações

A estratificação pode melhorar precisão quando estratos são homogêneos internamente. Se a fração coletada por estrato difere da população, a média simples da amostra pode ter viés.

In [3]:
e1=RNG.choice(pop[:800],40,replace=False);e2=RNG.choice(pop[800:],40,replace=False)
media_ponderada=.8*e1.mean()+.2*e2.mean()
print('não ponderada',np.r_[e1,e2].mean(),'ponderada',media_ponderada,'população',pop.mean())

não ponderada 14.051844731611066 ponderada 11.542869450406922 população 11.542216898008109


In [4]:
# Exportação reproduzível para outputs/ (dados e resultados didáticos)
pd.DataFrame({'amostra':['simples','seletiva','estratificada_nao_ponderada','estratificada_ponderada','populacao'], 'media':[simple.mean(),seletiva.mean(),np.r_[e1,e2].mean(),media_ponderada,pop.mean()]}).to_csv(OUTPUTS / '04_comparativo_amostragem.csv',index=False,encoding='utf-8-sig')
print('Resultados exportados em:', OUTPUTS)

Resultados exportados em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/03_statistics/outputs


## 3. Planejamento experimental e randomização

Defina unidade experimental, tratamento, desfecho, grupo controle e variável de confusão. Randomização reduz vieses sistemáticos *em expectativa*, mas não torna amostras pequenas perfeitamente balanceadas.

In [5]:
n=200;trat=np.zeros(n,dtype=bool);trat[RNG.choice(n,n//2,replace=False)]=True
base=RNG.normal(50,10,n);resultado=base+4*trat+RNG.normal(0,3,n) # efeito sintético conhecido
print('diferença observada de médias:',resultado[trat].mean()-resultado[~trat].mean())

diferença observada de médias: 5.407917014086415


In [6]:
# Exportação reproduzível para outputs/ (dados e resultados didáticos)
pd.DataFrame({'grupo':['controle','tratamento'],'n':[(~trat).sum(),trat.sum()],'media_resultado':[resultado[~trat].mean(),resultado[trat].mean()]}).to_csv(OUTPUTS / '04_resumo_experimento.csv',index=False,encoding='utf-8-sig')
print('Resultados exportados em:', OUTPUTS)

Resultados exportados em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/03_statistics/outputs


## 4. Experimento, observacional e causalidade

Uma associação em estudo observacional pode refletir confundimento. Em um ensaio randomizado, o efeito estimado é associado à atribuição aleatória quando respeitados o desenho, a medição e a análise.

In [7]:
print('Grupo tratado:',trat.sum(),'controle:',(~trat).sum())
assert trat.sum()==100

Grupo tratado: 100 controle: 100


### Exercício resolvido — Reprodutibilidade da amostra

**Solução comentada:**

In [8]:
v=np.arange(100)
a=np.random.default_rng(7).choice(v,10,replace=False)
b2=np.random.default_rng(7).choice(v,10,replace=False)
assert np.array_equal(a,b2)
print('amostra:',a)

amostra: [57 84 74 80 54 30 85  5 22 63]


**Interpretação:** Definir a semente torna o exemplo reproduzível, não garante representatividade.

## Erros comuns e limites

Não use tamanho da amostra como substituto de desenho amostral; distingue-se aleatorização da seleção e aleatorização da atribuição de tratamento.

## Consulta Rápida

`rng.choice(pop,n,replace=False)` · `rng.permutation(n)` · média estratificada = Σ peso_população × média_estrato.

## Autoavaliação

Antes de seguir, explique com suas palavras: **qual é a pergunta estatística, qual método responde a ela, quais pressupostos devem ser verificados e o que não podemos concluir?**

## Aprofundamento v2 — Erro amostral, viés e poder

**Representatividade** depende do mecanismo de seleção, e não apenas de `n`. Aumentar uma amostra selecionada por conveniência pode reduzir erro aleatório sem eliminar viés sistemático. Em experimentos, defina desfecho primário, diferença mínima relevante, unidade de randomização e plano de análise antes de observar resultados.

O cálculo a seguir é uma **aproximação de planejamento** para comparar duas médias independentes, tamanhos iguais, variâncias comuns e teste bilateral normal. Não substitui cálculos específicos para clusters, efeitos heterogêneos ou perdas.

In [9]:
from scipy import stats
alpha_planejamento=0.05; poder_desejado=0.80
z_alpha=stats.norm.ppf(1-alpha_planejamento/2)
z_beta=stats.norm.ppf(poder_desejado)
sigma_planejamento=10.0; delta_relevante=5.0
n_por_grupo = int(np.ceil(2*(z_alpha+z_beta)**2*sigma_planejamento**2/delta_relevante**2))
print('Amostra aproximada por grupo:', n_por_grupo)
assert n_por_grupo > 0

Amostra aproximada por grupo: 63


### Exercício resolvido

**Pergunta:** mantendo variância, alfa e poder, o que ocorre com `n` se a diferença mínima relevante cair pela metade?

**Solução:** nesta aproximação, `n` é proporcional a `1/delta²`; portanto, requer aproximadamente quatro vezes mais observações por grupo.